# Anima LoRA Trainer — v7（分桶与图片放大）

基于 v5 修改，底层是 [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts) 的 `anima_train_network.py`。模型仍是三个文件：DiT + Qwen3 文本编码器 + Qwen-Image VAE。


## v7 新增：分桶与训练时放大

- **Training Settings** 新增 `enable_bucket` 和 `allow_image_upscale`，默认都开启；小图可以等比放大到匹配的桶尺寸。
- 可调最小/最大桶边长、桶步长与缩放插值（默认 Lanczos）。Anima 要求分辨率和桶步长是 16 的倍数。
- `allow_image_upscale=True` 会写成真实后端参数 `bucket_no_upscale=False`；取消勾选即恢复 v6 的禁止放大行为。
- 新增只读的 **分桶与放大预览**：显示实际桶尺寸、将被放大的图片数量，以及原图 → 缩放 → 裁切尺寸示例。
- 这是训练时普通图片缩放，不是 AI 超分；不会覆盖云盘原图。关闭分桶时后端统一缩放/裁切，禁止放大选项不适用。
- 禁止放大时，后端会按原图尺寸生成桶，忽略最小/最大桶范围。修改这些设置后请重新运行 Settings 和 Training；续训旧 state 时应保持这些设置一致。

## v6 相对 v5 的改动

| 项目 | v5 | v6 |
|---|---|---|
| VAE 实现 | 3D VAE + `vae_chunk_size` / `vae_disable_cache` | **2D VAE（`qwen_image_vae_2d`）**：官方文档称编码约快 2 倍、峰值显存约 1/3，结果数值一致 |
| LLM Adapter 学习率 | 与全局相同 | 可单独设置（默认 5e-5，官方文档称调低更稳定） |
| `noise_offset` / `multires_noise_discount` | 写在配置里 | **已删除**，Anima 的 flow-matching 训练不使用这两项，写了也没有效果 |
| Warmup | 固定 100 步 | 总步数的 5%（步数少时不会大半程都在 warmup） |
| TensorBoard | 没有写日志，面板为空 | 写入 `output_directory/logs` |
| 图片计数 | 把所有非 `.txt` 文件都算作图片（缓存的 `.npz` 也会被算进去） | 只认图片扩展名 |
| 出图测试 | 每张图重新加载一次模型 | `--from_file` 一次加载，连续出 N 张 |
| 批量正则图 | 每张图重新加载一次模型 | 同上，一次加载 |
| 显存不够时 | 只能降 dim / 分辨率 | 新增 `blocks_to_swap`（把部分 Transformer block 放到 CPU） |
| 训练中采样 | 固定 768×768 | 尺寸、步数可调 |
| 断线续训 | 不支持 | 每个 epoch 保存 state，可自动接着最新 state 继续 |
| 日志 | 全部打印在单元格里（卡） | 写入文件，单元格只显示一行进度；loss 和采样图在 TensorBoard 看 |

## 使用步骤

1. 运行 **Setup**（挂载云盘、安装 sd-scripts）。
2. **Training Settings**：确认三个模型路径、数据集路径和超参数。
3. 可选但推荐：运行 **分桶与放大预览**，确认图片会如何缩放和裁切。
4. 运行 **Training**。
5. 可选：TensorBoard、出图测试、批量生成正则图。

## ⏯️ 断点续训

默认每个 epoch 会额外保存一个 `名称-0000NN-state` 文件夹（只保留最近 2 个）。Colab 断线后：

1. 重新运行 Setup 和 Training Settings，**数据集、batch、repeats、max_train_epochs 都不要改**；
2. 把 `resume_mode` 改成「自动：接着输出目录里最新的 state」；
3. 运行 Training，会从断掉的下一个 epoch 接着训练，学习率曲线也会接上。

没有 state 的旧 LoRA 可以用「从 LoRA 文件继续」：以它为起点再训练，但优化器和学习率从头开始。

## 数据集格式（平铺）

```
my_dataset/
  image001.png
  image001.txt    # 例：@cheese, 1boy, furry male, solo, ...
```

## ⚠️ Colab 时长

v5 测试中 1000 步需要 4 小时以上，Settings 单元格会估算步数，超过 1000 时给出提醒。

## Troubleshooting

- CUDA OOM：先把 `blocks_to_swap` 设为 8~16，再考虑降 `network_dim` / `resolution`。
- NaN loss：调低学习率。Anima 请保持 bf16（不要改 fp16）。
- 想用回 v5 的 VAE 行为：在设置里取消勾选 `use_2d_vae`。

In [ ]:
#@title Setup Cell — 挂载云盘 & 安装 sd-scripts

mount_drive = True #@param {type:"boolean"}
if mount_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive mounted at /content/drive")
    except Exception as e:
        print("⚠ Could not mount Google Drive:", e)

!if [ -d /content/sd-scripts ]; then echo 'sd-scripts already cloned'; else git clone https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts; fi
%cd /content/sd-scripts
!git checkout --detach cb70b96858477e716a2096292ab8aab433277f38
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install toml

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    if torch.cuda.get_device_capability(0)[0] < 8:
        print("⚠ 这张卡不原生支持 bf16（如 T4），Anima 训练会明显变慢。建议换 A100 / L4 运行时。")

In [ ]:
#@title Training Settings
#@markdown 设置训练参数。本单元格会估算总步数，超过 1000 步时给出提醒。

#@markdown ## Project Configuration
project_name = "cheese_sft_example" #@param {type:"string"}
image_directory = "/content/drive/MyDrive/lora_training/datasets/cheese_v24_train" #@param {type:"string"}
output_directory = "/content/drive/MyDrive/lora_training/output/cheese_sft_example" #@param {type:"string"}

#@markdown ## Training parameters
network_dim = 24 #@param {type:"integer"}
network_alpha = 24 #@param {type:"integer"}
learning_rate = 0.0001 #@param {type:"number"}
training_seed = 42 #@param {type:"integer"}
#@markdown LLM Adapter 层单独的学习率（官方文档：调低更稳定）。填 0 = 与 learning_rate 相同（等同 v5）。
llm_adapter_lr = 0 #@param {type:"number"}
max_train_epochs = 20 #@param {type:"integer"}
train_batch_size = 4 #@param {type:"integer"}
gradient_accumulation_steps = 1 #@param {type:"integer"}

#@markdown ## Dataset Settings
resolution = 768 #@param {type:"integer"}
repeats = 1 #@param {type:"integer"}
caption_dropout = 0.05 #@param {type:"slider", min:0, max:1, step:0.05}

#@markdown ### 分桶与训练时图片放大
#@markdown 分桶按图片宽高比分组；勾选后竖图、横图不会全部裁成正方形。
enable_bucket = True #@param {type:"boolean"}
#@markdown 允许小图在训练时放大到匹配的桶尺寸。不会修改云盘原图；这是普通插值缩放，不是 AI 超分。
allow_image_upscale = True #@param {type:"boolean"}
#@markdown 下列最小/最大边长在「开启分桶且允许放大」时用于生成桶；禁止放大时后端按原图尺寸生成桶，会忽略这两个范围。
min_bucket_reso = 256 #@param {type:"integer"}
max_bucket_reso = 2048 #@param {type:"integer"}
#@markdown Anima 要求步长为 16 的倍数，64 是常用值。
bucket_reso_steps = 64 #@param [16, 32, 64, 128] {type:"raw"}
image_resize_interpolation = "lanczos" #@param ["lanczos", "bicubic", "bilinear", "area"]
#@markdown 关闭分桶时，后端会统一等比缩放并裁切至 resolution×resolution；此时 allow_image_upscale 无效，小图仍可能被放大。

#@markdown ## 显存 / 速度
#@markdown 使用 2D Qwen-Image VAE（更快、更省显存，结果与 3D 版数值一致）。
use_2d_vae = True #@param {type:"boolean"}
#@markdown 显存不够时把部分 Transformer block 放到 CPU。0 = 关闭；越大越省显存但越慢（28 层模型最大 26）。
blocks_to_swap = 0 #@param {type:"integer"}

#@markdown ## ⏯️ 断点续训
#@markdown 每个 epoch 额外保存训练状态（LoRA + 优化器 + 学习率进度），保存在输出目录的 `名称-000005-state` 文件夹里。
save_training_state = True #@param {type:"boolean"}
#@markdown 只保留最近 N 个 state，避免占满云盘。
keep_last_n_states = 2 #@param {type:"integer"}
#@markdown **续训方式**
#@markdown - `不续训`：从头开始
#@markdown - `自动：接着输出目录里最新的 state`：Colab 断线后重跑就选这个（数据集、batch、repeats 不要改）
#@markdown - `指定 state 文件夹`：在 `resume_path` 填某个 `xxx-000005-state` 文件夹
#@markdown - `从 LoRA 文件继续`：在 `resume_path` 填 `.safetensors`，以它为起点训练（优化器和学习率从头开始，dim/alpha 自动跟随该 LoRA）
resume_mode = "不续训" #@param ["不续训", "自动：接着输出目录里最新的 state", "指定 state 文件夹", "从 LoRA 文件继续"]
resume_path = "" #@param {type:"string"}

#@markdown ## Custom Base Model Selection (Optional)
#@markdown 留空则使用下面默认的云盘 ComfyUI 路径。
custom_dit_model = "" #@param {type:"string"}
custom_qwen3_model = "" #@param {type:"string"}
custom_vae_model = "" #@param {type:"string"}

import os, math as _math

DRIVE_DIT   = '/content/drive/MyDrive/ComfyUI/models/diffusion_models/waiANIMA_v10Base10.safetensors'
DRIVE_QWEN3 = '/content/drive/MyDrive/ComfyUI/models/text_encoders/waiANIMA_v10Base10_txt.safetensors'
DRIVE_VAE   = '/content/drive/MyDrive/ComfyUI/models/vae/qwenImage_qwenImageVAE.safetensors'

DIT_MODEL   = custom_dit_model.strip()   or DRIVE_DIT
QWEN3_MODEL = custom_qwen3_model.strip() or DRIVE_QWEN3
VAE_MODEL   = custom_vae_model.strip()   or DRIVE_VAE

IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.gif')

def count_images(d):
    return len([f for f in os.listdir(d)
                if f.lower().endswith(IMAGE_EXTS) and os.path.isfile(os.path.join(d, f))])


def build_dataset_resize_options(resolution, enable_bucket, allow_image_upscale,
                                 min_bucket_reso, max_bucket_reso, bucket_reso_steps,
                                 image_resize_interpolation):
    """校验界面参数，并生成 sd-scripts 数据集配置中的真实字段。"""
    for name, value in [('resolution', resolution), ('min_bucket_reso', min_bucket_reso),
                        ('max_bucket_reso', max_bucket_reso), ('bucket_reso_steps', bucket_reso_steps)]:
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f'{name} 必须是正整数，当前为 {value!r}')
    if not isinstance(enable_bucket, bool) or not isinstance(allow_image_upscale, bool):
        raise ValueError('enable_bucket 和 allow_image_upscale 必须为布尔值，请使用勾选框。')
    if resolution % 16:
        raise ValueError('Anima 的 resolution 必须是 16 的倍数。')
    if bucket_reso_steps % 16:
        raise ValueError('Anima 的 bucket_reso_steps 必须是 16 的倍数，例如 16、32、64。')
    if enable_bucket:
        if min_bucket_reso > max_bucket_reso:
            raise ValueError('min_bucket_reso 不能大于 max_bucket_reso。')
        if not (min_bucket_reso <= resolution <= max_bucket_reso):
            raise ValueError('请设置 min_bucket_reso <= resolution <= max_bucket_reso。')
        if min_bucket_reso % bucket_reso_steps or max_bucket_reso % bucket_reso_steps:
            raise ValueError('min_bucket_reso 和 max_bucket_reso 必须是 bucket_reso_steps 的倍数。')
    if image_resize_interpolation not in ('lanczos', 'bicubic', 'bilinear', 'area'):
        raise ValueError('不支持的图片缩放插值方式。')
    return {
        'resolution': resolution,
        'enable_bucket': enable_bucket,
        'bucket_no_upscale': not allow_image_upscale,
        'bucket_reso_steps': bucket_reso_steps,
        'min_bucket_reso': min_bucket_reso,
        'max_bucket_reso': max_bucket_reso,
        'resize_interpolation': image_resize_interpolation,
    }

dataset_resize_options = build_dataset_resize_options(
    resolution, enable_bucket, allow_image_upscale, min_bucket_reso,
    max_bucket_reso, bucket_reso_steps, image_resize_interpolation,
)

print("="*60)
print("TRAINING CONFIGURATION (Anima v7)")
print("="*60)
print(f"Project Name:      {project_name}")
print(f"Image Directory:   {image_directory}")
print(f"Output Directory:  {output_directory}")
print(f"Network Dim/Alpha: {network_dim} / {network_alpha}")
print(f"Learning Rate:     {learning_rate}  (LLM adapter: {llm_adapter_lr if llm_adapter_lr > 0 else 'same'})")
print(f"Max Epochs:        {max_train_epochs}")
print(f"Batch Size:        {train_batch_size} × grad_accum {gradient_accumulation_steps}")
print(f"Resolution:        {resolution}px")
print(f"Bucketing:         {enable_bucket}")
print(f"Allow upscale:     {allow_image_upscale}  (bucket_no_upscale={not allow_image_upscale})")
print(f"Bucket range/step: {min_bucket_reso}..{max_bucket_reso} / {bucket_reso_steps}")
print(f"Resize method:     {image_resize_interpolation}")
if not enable_bucket:
    print('ℹ 分桶已关闭：统一缩放并裁切到固定分辨率；禁止放大开关在此模式下不生效。')
elif not allow_image_upscale:
    print('ℹ 禁止放大：桶按原图尺寸生成，min/max_bucket_reso 范围由后端忽略。')
print(f"Repeats:           {repeats}")
print(f"Caption Dropout:   {caption_dropout}")
print(f"2D VAE:            {use_2d_vae}")
print(f"Blocks to swap:    {blocks_to_swap}")
for _n, _p in [('DiT', DIT_MODEL), ('Qwen3', QWEN3_MODEL), ('VAE', VAE_MODEL)]:
    print(f"{_n + ' Model:':<19}{_p}  {'✓' if os.path.exists(_p) else '❌ 找不到'}")
print("="*60)

print()
print("── Step Estimate（粗略，分桶尾批可能增加步数）─────────")
try:
    _n = count_images(image_directory)
    _eff = int(train_batch_size) * int(gradient_accumulation_steps)
    _spe = _math.ceil(_n * int(repeats) / _eff)
    _tot = _spe * int(max_train_epochs)
    print(f"  Images found:        {_n}")
    print(f"  Steps per epoch:     {_spe}  ({_n} images × {repeats} repeats / {_eff} eff. batch)")
    print(f"  Total steps:         {_tot}  ({_spe} × {max_train_epochs} epochs)")
    if _tot > 1000:
        print()
        print("  ⚠️  WARNING: Total steps exceed 1000 — 测试中 1000 步需 4 小时以上，Colab 可能中途断开。")
        print(f"    • Lower epochs   (current: {max_train_epochs})  →  try {max(1000 // _spe, 1)}")
        print(f"    • Lower repeats  (current: {repeats})")
    else:
        print("  ✓ Steps look good — within the recommended < 1000 limit.")
except FileNotFoundError:
    print("  (image_directory not found yet — steps will be checked at runtime)")
except Exception as _e:
    print(f"  Could not estimate steps: {_e}")
print("──────────────────────────────────────────────────────")

In [ ]:
#@title 分桶与放大预览（可选，只读）
#@markdown 先运行 Setup 和 Training Settings。本单元格使用 sd-scripts 自己的 BucketManager 检查训练图的实际目标尺寸，不写图片、不加载模型、不开始训练。
run_dataset_preview = True #@param {type:"boolean"}
preview_example_count = 12 #@param {type:"integer"}

from collections import Counter
import math, os, sys
from PIL import Image as PreviewImage

def preview_dataset_buckets(image_dir, *, max_examples=12):
    if not os.path.isdir(image_dir):
        raise FileNotFoundError(f'训练图目录不存在: {image_dir}')
    sd_path = '/content/sd-scripts'
    if sd_path not in sys.path:
        sys.path.insert(0, sd_path)
    try:
        from library.train_util import BucketManager
    except ImportError as exc:
        raise RuntimeError('请先成功运行 Setup，再运行此预览。') from exc
    opts = build_dataset_resize_options(
        resolution, enable_bucket, allow_image_upscale, min_bucket_reso,
        max_bucket_reso, bucket_reso_steps, image_resize_interpolation,
    )
    target = (opts['resolution'], opts['resolution'])
    if opts['enable_bucket']:
        manager = BucketManager(opts['bucket_no_upscale'], target, opts['min_bucket_reso'],
                                opts['max_bucket_reso'], opts['bucket_reso_steps'])
        if not opts['bucket_no_upscale']:
            manager.make_buckets()
    else:
        manager = BucketManager(False, target, None, None, None)
        manager.set_predefined_resos([target])
    names = sorted(f for f in os.listdir(image_dir)
                   if f.lower().endswith(IMAGE_EXTS) and os.path.isfile(os.path.join(image_dir, f)))
    if not names:
        raise ValueError('训练目录里没有找到图片。')
    buckets, actions, examples, errors = Counter(), Counter(), [], []
    for name in names:
        try:
            with PreviewImage.open(os.path.join(image_dir, name)) as im:
                width, height = im.size
            if opts['enable_bucket'] and opts['bucket_no_upscale'] and min(width, height) < opts['bucket_reso_steps']:
                raise ValueError('短边不足一个桶步长；请允许放大或移除此小图')
            bucket, resized, _ = manager.select_bucket(width, height)
            if min(bucket) <= 0:
                raise ValueError('目标桶尺寸为零')
            action = ('放大' if resized[0] > width or resized[1] > height else
                      '缩小' if resized[0] < width or resized[1] < height else '不缩放')
            actions[action] += 1
            buckets[tuple(bucket)] += 1
            if len(examples) < max(0, int(max_examples)):
                examples.append({'file': name, 'original': (width, height),
                                 'resized': tuple(resized), 'bucket': tuple(bucket), 'action': action})
        except Exception as exc:
            errors.append(f'{name}: {exc}')
    if errors:
        raise ValueError('以下图片无法预览，请先处理再训练：\n' + '\n'.join(errors[:10]))
    print(f"训练图 {len(names)} 张；{len(buckets)} 个桶；放大 {actions['放大']} 张，缩小 {actions['缩小']} 张，不缩放 {actions['不缩放']} 张。")
    print('桶尺寸（宽×高）      图片数')
    for (bw, bh), count in sorted(buckets.items()):
        print(f'{bw:5d}×{bh:<5d}          {count}')
    print('\n示例：原图 → 等比缩放后 → 目标桶（超出部分会裁切）')
    for row in examples:
        print(f"{row['file']}: {row['original']} → {row['resized']} → {row['bucket']}  {row['action']}")
    if int(train_batch_size) <= 0 or int(gradient_accumulation_steps) <= 0 or int(repeats) <= 0:
        raise ValueError('batch、梯度累积步数和 repeats 必须大于 0。')
    micro_batches = sum(math.ceil(count * int(repeats) / int(train_batch_size)) for count in buckets.values())
    steps_per_epoch = math.ceil(micro_batches / int(gradient_accumulation_steps))
    print(f'\n单卡训练图估算：每轮 {steps_per_epoch} 个优化器步，合计 {steps_per_epoch * int(max_train_epochs)} 步。')
    print('此估算不含训练单元格中另行指定的正则图，以训练器最终日志为准。')
    return {'image_count': len(names), 'buckets': dict(buckets), 'actions': dict(actions),
            'examples': examples, 'steps_per_epoch': steps_per_epoch}

if run_dataset_preview:
    dataset_preview = preview_dataset_buckets(image_directory, max_examples=preview_example_count)
else:
    print('已跳过分桶预览。')


In [ ]:
#@title Training Cell — 生成配置并开始训练（含训练中采样 & 正则图支持）

import os, re, sys, json, time, html, shlex, shutil, toml, math, subprocess
import numpy as np
from PIL import Image as PILImage
from IPython.display import display, HTML
from datetime import datetime

os.environ.setdefault('PYTHONUNBUFFERED', '1')

g = globals()
DIT_MODEL   = g.get('DIT_MODEL')
QWEN3_MODEL = g.get('QWEN3_MODEL')
VAE_MODEL   = g.get('VAE_MODEL')

CONFIG_DIR = '/content/lora_training/configs'
os.makedirs(CONFIG_DIR, exist_ok=True)

#@markdown ### 🔍 训练中采样（可选）
enable_sample_generation = True #@param {type:"boolean"}
sample_prompt_base = "@cheese, masterpiece, best quality, score_9, score_8, score_7, 1boy, furry male, solo, nude" #@param {type:"string"}
sample_negative_prompt = "worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, lowres, censor" #@param {type:"string"}
sample_interval_epochs = 1 #@param {type:"integer"}
sample_sampler = "euler_a" #@param ["euler_a", "ddim", "heun", "dpm_2", "dpm_2_a", "lms"]
sample_cfg_scale = 5.0 #@param {type:"number"}
sample_steps = 20 #@param {type:"integer"}
sample_width = 768 #@param {type:"integer"}
sample_height = 768 #@param {type:"integer"}

#@markdown ### 🛡️ 正则图数据集（可选，留空不使用）
reg_data_directory = "" #@param {type:"string"}

#@markdown ### 📝 日志显示
#@markdown 完整日志写到本地文件，单元格里只显示一行进度；loss 曲线和每个 epoch 的采样图在 TensorBoard 里看。
launch_tensorboard = True #@param {type:"boolean"}
status_interval_sec = 15 #@param {type:"integer"}

IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.gif')

def list_images(d):
    return [f for f in os.listdir(d) if f.lower().endswith(IMAGE_EXTS) and os.path.isfile(os.path.join(d, f))]

def create_sample_prompt_file(project_name, base_prompt, negative_prompt, cfg_scale, steps, w, h):
    path = os.path.join(CONFIG_DIR, f"{project_name}_sample_prompt.txt")
    perspectives = [
        "standing, full body, looking at viewer, front view, cinematic lighting, soft shadows, safe",
        "standing, full body, profile, side view, cinematic lighting, soft shadows, safe",
        "standing, full body, back view, looking back at viewer, cinematic lighting, soft shadows, safe",
        "dynamic action pose, mid-action, running, jumping, energetic, cinematic lighting, soft shadows, safe",
    ]
    lines = [f"{base_prompt}, {p} --n {negative_prompt} --w {w} --h {h} --l {cfg_scale} --s {steps} --d 42\n"
             for p in perspectives]
    with open(path, 'w', encoding='utf-8') as f:
        f.writelines(lines)
    print(f"✓ Created sample prompt file ({len(lines)} prompts, {w}x{h}): {path}")
    return path

def create_training_config(project_name, output_dir, dit, qwen3, vae, *, network_dim, network_alpha,
                           learning_rate, llm_adapter_lr, max_train_epochs, train_batch_size, grad_accum,
                           use_2d_vae, blocks_to_swap,
                           sample_prompt_path=None, sample_every_n_epochs=1, sample_sampler_val='euler_a', extra_cfg=None):
    os.makedirs(output_dir, exist_ok=True)
    current_date = datetime.now().strftime('%Y-%m-%d')
    config_path = os.path.join(CONFIG_DIR, f"{project_name}_training_config_{current_date}.toml")

    cfg = {
        'pretrained_model_name_or_path': dit,
        'qwen3': qwen3,
        'vae': vae,
        'network_module': 'networks.lora_anima',
        'network_dim': int(network_dim),
        'network_alpha': int(network_alpha),
        'network_train_unet_only': True,
        'learning_rate': float(learning_rate),
        'optimizer_type': 'AdamW8bit',
        'optimizer_args': ['weight_decay=0.1', 'betas=[0.9, 0.99]'],
        'lr_scheduler': 'cosine_with_restarts',
        'lr_scheduler_num_cycles': 1,
        'lr_warmup_steps': 0.05,           # 总步数的 5%
        'max_train_epochs': int(max_train_epochs),
        'train_batch_size': int(train_batch_size),
        'gradient_accumulation_steps': int(grad_accum),
        'max_grad_norm': 1.0,
        'seed': int(globals().get('training_seed', 42)),
        'timestep_sampling': 'sigmoid',
        'discrete_flow_shift': 1.0,
        'qwen3_max_token_length': 512,
        't5_max_token_length': 512,
        'mixed_precision': 'bf16',
        'attn_mode': 'torch',
        'gradient_checkpointing': True,
        'cache_latents': True,
        'cache_text_encoder_outputs': True,   # Anima 的 TE 缓存支持 caption_dropout
        'max_data_loader_n_workers': 2,
        'persistent_data_loader_workers': True,
        'output_dir': output_dir,
        'output_name': project_name,
        'save_model_as': 'safetensors',
        'save_precision': 'bf16',
        'save_every_n_epochs': 1,
        'shuffle_caption': False,
        'caption_extension': '.txt',
        'logging_dir': TB_ROOT,
        'log_with': 'tensorboard',
        'log_prefix': project_name,
        'training_comment': f'Anima LoRA v7 - {current_date}',
    }
    if use_2d_vae:
        cfg['qwen_image_vae_2d'] = True
    else:
        cfg['vae_chunk_size'] = 64
        cfg['vae_disable_cache'] = True
    if llm_adapter_lr and float(llm_adapter_lr) > 0:
        cfg['network_args'] = [f"network_reg_lrs=.*llm_adapter.*={float(llm_adapter_lr)}"]
    if blocks_to_swap and int(blocks_to_swap) > 0:
        cfg['blocks_to_swap'] = int(blocks_to_swap)
    if sample_prompt_path:
        cfg['sample_prompts'] = sample_prompt_path
        cfg['sample_every_n_epochs'] = int(sample_every_n_epochs)
        cfg['sample_sampler'] = sample_sampler_val

    if extra_cfg:
        cfg.update(extra_cfg)
    with open(config_path, 'w') as f:
        toml.dump(cfg, f)
    print(f"✓ Created training config: {os.path.basename(config_path)}")
    return config_path

def create_dataset_config(project_name, image_dir, reg_dir="", resolution=768, repeats=2, caption_dropout_rate=0.05,
                          *, enable_bucket=True, allow_image_upscale=True, min_bucket_reso=256,
                          max_bucket_reso=2048, bucket_reso_steps=64, image_resize_interpolation='lanczos'):
    resize_options = build_dataset_resize_options(
        resolution, enable_bucket, allow_image_upscale, min_bucket_reso,
        max_bucket_reso, bucket_reso_steps, image_resize_interpolation,
    )
    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Image directory not found: {image_dir}")
    if not list_images(image_dir):
        raise ValueError(f"No image files found in {image_dir}")

    subsets = [{
        'num_repeats': int(repeats),
        'image_dir': image_dir,
        'caption_extension': '.txt',
        'caption_dropout_rate': float(caption_dropout_rate),
    }]
    if reg_dir.strip() and os.path.exists(reg_dir.strip()):
        reg_path = reg_dir.strip()
        reg_files = list_images(reg_path)
        if reg_files:
            subsets.append({'num_repeats': 1, 'image_dir': reg_path, 'is_reg': True,
                            'caption_extension': '.txt', 'caption_dropout_rate': 0.0})
            print(f"✓ Loaded regularization images from: {reg_path} (Found {len(reg_files)} images)")
        else:
            print(f"⚠ Regularization folder '{reg_path}' is empty. Skipping reg images.")

    current_date = datetime.now().strftime('%Y-%m-%d')
    config_path = os.path.join(CONFIG_DIR, f"{project_name}_dataset_config_{current_date}.toml")
    dataset_config = {
        'general': resize_options,
        'datasets': [{'resolution': int(resolution), 'subsets': subsets}],
    }
    with open(config_path, 'w') as f:
        toml.dump(dataset_config, f)
    print(f"✓ Created dataset config: {os.path.basename(config_path)}")
    return config_path

OOM_HINT = '💡 CUDA OOM：先试 blocks_to_swap=8~16，再降 network_dim / resolution。'

TB_ROOT  = '/content/lora_training/tb'          # TensorBoard 日志（本地磁盘，读写快）
LOG_ROOT = '/content/lora_training/train_logs'  # 完整文本日志

def _tail_status(log_path, max_bytes=65536):
    """从日志尾部取当前进度：优先 tqdm 的 steps 行，否则取最后一行非空输出。"""
    try:
        with open(log_path, 'rb') as f:
            f.seek(0, 2); size = f.tell()
            f.seek(max(0, size - max_bytes))
            chunk = f.read().decode('utf-8', 'ignore')
    except FileNotFoundError:
        return ''
    segs = [s.strip() for s in re.split(r'[\r\n]+', chunk) if s.strip()]
    for s in reversed(segs):
        if s.startswith('steps:'):
            return s
    return segs[-1][-300:] if segs else ''

def _push_samples(writer, sample_dir, seen):
    """把新生成的采样图写进 TensorBoard（IMAGES 页，按 epoch 拖动查看）。"""
    if writer is None or not os.path.isdir(sample_dir):
        return 0
    n = 0
    for fn in sorted(os.listdir(sample_dir)):
        if not fn.endswith('.png') or fn in seen:
            continue
        m = re.search(r'_(?:e(\d{6})|(\d{6}))_(\d{2})_', fn)
        if not m:
            continue
        seen.add(fn)
        step = int(m.group(1) or m.group(2)); idx = int(m.group(3))
        try:
            img = PILImage.open(os.path.join(sample_dir, fn)).convert('RGB')
            img.thumbnail((640, 640))
            writer.add_image(f'samples/prompt_{idx:02d}', np.asarray(img), global_step=step, dataformats='HWC')
            n += 1
        except Exception as e:
            print(f'⚠ 采样图写入 TensorBoard 失败 {fn}: {e}')
    if n:
        writer.flush()
    return n

def run_training(cmd_list, project_name, output_dir):
    os.makedirs(TB_ROOT, exist_ok=True); os.makedirs(LOG_ROOT, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path = os.path.join(LOG_ROOT, f'{project_name}_{ts}.log')

    if launch_tensorboard:
        try:
            from tensorboard import notebook as _tbnb
            _tbnb.start(f'--logdir {TB_ROOT} --reload_interval 15')
        except Exception as e:
            print(f'⚠ TensorBoard 启动失败（可稍后运行 TensorBoard 单元格）: {e}')

    writer = None
    try:
        from torch.utils.tensorboard import SummaryWriter
        writer = SummaryWriter(log_dir=os.path.join(TB_ROOT, f'{project_name}_{ts}_samples'))
    except Exception as e:
        print(f'⚠ 无法把采样图写入 TensorBoard: {e}')

    env = dict(os.environ, PYTHONUNBUFFERED='1', TQDM_MININTERVAL='5')
    print('Executing:', ' '.join(shlex.quote(c) for c in cmd_list))
    print(f'📝 完整日志: {log_path}')
    print(f'📊 TensorBoard 日志: {TB_ROOT}')
    print()

    lf = open(log_path, 'w', encoding='utf-8')
    proc = subprocess.Popen(cmd_list, stdout=lf, stderr=subprocess.STDOUT, cwd='/content/sd-scripts', env=env)
    handle = display(HTML('<pre>启动中…</pre>'), display_id=True)
    sample_dir = os.path.join(output_dir, 'sample')
    seen, n_samples, t0 = set(), 0, time.time()
    try:
        while True:
            rc = proc.poll()
            n_samples += _push_samples(writer, sample_dir, seen)
            el = int(time.time() - t0)
            status = _tail_status(log_path)
            head = f'⏱ {el // 3600:d}:{el % 3600 // 60:02d}:{el % 60:02d}   🖼 采样图 {n_samples} 张已送入 TensorBoard'
            if handle is not None:
                handle.update(HTML(f'<pre>{head}\n{html.escape(status)}</pre>'))
            else:
                print(f'{head} | {status}', flush=True)
            if rc is not None:
                break
            time.sleep(max(3, int(status_interval_sec)))
    except KeyboardInterrupt:
        proc.kill(); proc.wait()
        print('⏹ 已手动中断训练。')
        raise
    finally:
        lf.close()
        if writer is not None:
            _push_samples(writer, sample_dir, seen)
            writer.close()
        # 把日志备份到云盘输出目录，运行时断开后也能看
        try:
            dst = os.path.join(output_dir, 'logs')
            os.makedirs(dst, exist_ok=True)
            shutil.copy2(log_path, dst)
            shutil.copytree(TB_ROOT, os.path.join(dst, 'tensorboard'), dirs_exist_ok=True)
            print(f'💾 日志已备份到: {dst}')
        except Exception as e:
            print(f'⚠ 日志备份失败: {e}')

    if rc == 0:
        print('\n✓ Training completed successfully!')
        return True

    print(f'\n✗ Training failed (exit code: {rc}). 日志最后 60 行：\n' + '-' * 60)
    with open(log_path, encoding='utf-8', errors='ignore') as f:
        lines = [l for l in re.split(r'[\r\n]+', f.read()) if l.strip() and not l.lstrip().startswith('steps:')]
    print('\n'.join(lines[-60:]))
    print('-' * 60)
    if any('out of memory' in l.lower() for l in lines[-200:]):
        print(OOM_HINT)
    return False

def find_latest_state(output_dir, project_name):
    if not os.path.isdir(output_dir):
        return None
    cands = [os.path.join(output_dir, d) for d in os.listdir(output_dir)
             if d.startswith(project_name + '-') and d.endswith('-state')
             and os.path.isdir(os.path.join(output_dir, d))]
    return max(cands, key=os.path.getmtime) if cands else None

def build_resume_cfg(output_dir, project_name, total_epochs):
    g = globals()
    cfg = {}
    if g.get('save_training_state', True):
        cfg['save_state'] = True
        n = int(g.get('keep_last_n_states', 2) or 0)
        if n > 0:
            cfg['save_last_n_epochs_state'] = n

    mode = g.get('resume_mode', '不续训')
    path = (g.get('resume_path') or '').strip()
    print()
    if mode.startswith('自动'):
        st = find_latest_state(output_dir, project_name)
        if st:
            cfg['resume'] = st
        else:
            print('⏯️ 输出目录里没有找到 state，从头开始训练。')
    elif mode.startswith('指定'):
        if not path or not os.path.isdir(path):
            raise SystemExit(f'⏯️ state 文件夹不存在: {path!r}')
        cfg['resume'] = path
    elif mode.startswith('从 LoRA'):
        if not path or not os.path.isfile(path):
            raise SystemExit(f'⏯️ LoRA 文件不存在: {path!r}')
        cfg['network_weights'] = path
        cfg['dim_from_weights'] = True
        print(f'⏯️ 以 LoRA 为起点继续训练: {path}')
        print('   优化器和学习率从头开始，epoch 从 1 重新计数；dim/alpha 跟随该 LoRA。')
    else:
        print('⏯️ 不续训，从头开始。')

    if 'resume' in cfg:
        st = cfg['resume']
        done_epoch = None
        try:
            with open(os.path.join(st, 'train_state.json')) as f:
                done_epoch = json.load(f).get('current_epoch')
        except Exception:
            pass
        print(f'⏯️ 从 state 续训: {st}')
        if done_epoch is not None:
            if done_epoch >= total_epochs:
                raise SystemExit(f'⏯️ 这个 state 已经训练完 {done_epoch} 个 epoch。想再多训练，'
                                 f'请把 max_train_epochs 调大（当前 {total_epochs}），或改用「从 LoRA 文件继续」。')
            print(f'   已完成 {done_epoch} / {total_epochs} 个 epoch，将从第 {done_epoch + 1} 个 epoch 继续。')
        else:
            print('   ⚠ 没找到 train_state.json，无法确认进度（sd-scripts 仍会尝试加载）。')
    return cfg

def main():
    g = globals()
    project_name     = g.get('project_name', 'my_lora')
    image_directory  = g.get('image_directory')
    output_directory = g.get('output_directory', '/content/lora_output')
    repeats          = g.get('repeats', 2)
    batch            = int(g.get('train_batch_size', 1))
    grad_accum       = int(g.get('gradient_accumulation_steps', 1))
    epochs           = int(g.get('max_train_epochs', 10))

    if not image_directory or not os.path.exists(image_directory):
        raise SystemExit(f"Image directory not found: {image_directory}")

    n_images = len(list_images(image_directory))
    total_steps = math.ceil(n_images * int(repeats) / (batch * grad_accum)) * epochs
    print(f"\nStep check: {n_images} images × {repeats} repeats × {epochs} epochs / {batch * grad_accum} = {total_steps} steps")

    print("\nValidating models...")
    ok = True
    for name, path in [('DiT', DIT_MODEL), ('Qwen3', QWEN3_MODEL), ('VAE', VAE_MODEL)]:
        if path and os.path.exists(path):
            print(f"  ✓ {name}: {path}")
        else:
            print(f"  ✗ {name} missing: {path}")
            ok = False
    if not ok:
        raise SystemExit("Required models missing — check Google Drive paths.")

    sample_prompt_path = None
    if enable_sample_generation and sample_prompt_base.strip():
        sample_prompt_path = create_sample_prompt_file(project_name, sample_prompt_base, sample_negative_prompt,
                                                       sample_cfg_scale, sample_steps, sample_width, sample_height)

    resume_cfg = build_resume_cfg(output_directory, project_name, epochs)

    train_cfg = create_training_config(
        project_name, output_directory, DIT_MODEL, QWEN3_MODEL, VAE_MODEL,
        network_dim=g.get('network_dim', 24), network_alpha=g.get('network_alpha', 24),
        learning_rate=g.get('learning_rate', 1e-4), llm_adapter_lr=g.get('llm_adapter_lr', 0),
        max_train_epochs=epochs, train_batch_size=batch, grad_accum=grad_accum,
        use_2d_vae=g.get('use_2d_vae', True), blocks_to_swap=g.get('blocks_to_swap', 0),
        sample_prompt_path=sample_prompt_path, sample_every_n_epochs=sample_interval_epochs,
        sample_sampler_val=sample_sampler, extra_cfg=resume_cfg,
    )
    dataset_cfg = create_dataset_config(
        project_name, image_directory, reg_dir=reg_data_directory,
        resolution=g.get('resolution', 768), repeats=repeats, caption_dropout_rate=g.get('caption_dropout', 0.05),
        enable_bucket=g.get('enable_bucket', True),
        allow_image_upscale=g.get('allow_image_upscale', True),
        min_bucket_reso=g.get('min_bucket_reso', 256),
        max_bucket_reso=g.get('max_bucket_reso', 2048),
        bucket_reso_steps=g.get('bucket_reso_steps', 64),
        image_resize_interpolation=g.get('image_resize_interpolation', 'lanczos'),
    )

    cmd = [
        'accelerate', 'launch',
        '--num_cpu_threads_per_process', '1',
        '--mixed_precision', 'bf16',
        '/content/sd-scripts/anima_train_network.py',
        '--config_file', train_cfg,
        '--dataset_config', dataset_cfg,
    ]
    success = run_training(cmd, project_name, output_directory)

    print('\n=== Training Summary ===')
    print(f'Project: {project_name}')
    print('Status:', '✓ Success' if success else '✗ Failed')
    if success:
        print(f'Trained LoRA saved to: {output_directory}')

try:
    main()
except SystemExit as e:
    print(e)
except Exception as e:
    print('Fatal error:', e)
    import traceback; traceback.print_exc()

### 📊 TensorBoard
训练单元格会自动打开 TensorBoard（SCALARS 看 loss，IMAGES 看每个 epoch 的采样图）。日志在本地 `/content/lora_training/tb`，训练结束后自动备份到云盘 `output_directory/logs/`。

In [ ]:
#@title 📊 Launch TensorBoard
#@markdown 训练单元格默认已自动打开 TensorBoard；运行时重启后或想单独查看时用这个。
#@markdown 本地日志不存在时（例如运行时已断开重连），会自动改看云盘 `output_directory/logs/tensorboard` 里的备份。
import os
from tensorboard import notebook as _tbnb
g = globals()
TB_ROOT = '/content/lora_training/tb'
output_directory = g.get('output_directory', '')
log_dir = TB_ROOT
if not os.path.isdir(TB_ROOT) or not os.listdir(TB_ROOT):
    backup = os.path.join(output_directory, 'logs', 'tensorboard')
    if os.path.isdir(backup):
        log_dir = backup
print(f"🔍 TensorBoard 正在监控: {log_dir}")
print("   SCALARS 页看 loss / 学习率，IMAGES 页看每个 epoch 的采样图（拖动滑块切换 epoch）。")
_tbnb.start(f'--logdir "{log_dir}" --reload_interval 15')

In [ ]:
#@title 🧪 出图测试（可选加载 LoRA）
# v6：用 --from_file 一次加载模型，连续生成 num_images 张。

prompt = "@cheese, masterpiece, best quality, score_9, score_8, score_7, 1boy, solo, dynamic pose, standing, 3/4 view, full body portrait, character design sheet, official art, looking at viewer, yellow paw pads, solid white background, simple background, aesthetic lighting, soft shadows, nude, safe" #@param {type:"string"}
negative_prompt = "worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, lowres, censor, cluttered background, complex environment, photo, photorealistic, 3d render, digital art, flat shading, smooth textures, symmetric pose, rigid posture, holding weapon, multi-angle, expression sheet" #@param {type:"string"}
width = 768 #@param {type:"integer"}
height = 1024 #@param {type:"integer"}
infer_steps = 28 #@param {type:"integer"}
cfg_scale = 5 #@param {type:"number"}

#@markdown ### 🎲 Generation Settings
num_images = 2 #@param {type:"integer"}
seed = -1 #@param {type:"integer"}
#@markdown *seed = -1 时每张随机；固定 seed 时依次使用 seed, seed+1, seed+2 …*

#@markdown ### ⚡ Performance Settings
#@markdown T4 上请保持 `use_fp8` 关闭（T4 没有 FP8 硬件，会慢 3-4 倍）。
use_fp8 = False #@param {type:"boolean"}

#@markdown ### 🌸 LoRA Settings
use_lora = True #@param {type:"boolean"}
#@markdown 留空则自动使用输出目录中最新的 LoRA。
custom_lora_path = "" #@param {type:"string"}
lora_multiplier = 1 #@param {type:"slider", min:0, max:2, step:0.05}

import os, sys, glob, random, subprocess
from IPython.display import Image, display

os.environ.setdefault('PYTHONUNBUFFERED', '1')

g = globals()
DIT_MODEL        = g.get('DIT_MODEL', '/content/drive/MyDrive/ComfyUI/models/diffusion_models/waiANIMA_v10Base10.safetensors')
QWEN3_MODEL      = g.get('QWEN3_MODEL', '/content/drive/MyDrive/ComfyUI/models/text_encoders/waiANIMA_v10Base10_txt.safetensors')
VAE_MODEL        = g.get('VAE_MODEL', '/content/drive/MyDrive/ComfyUI/models/vae/qwenImage_qwenImageVAE.safetensors')
USE_2D_VAE       = g.get('use_2d_vae', True)
output_directory = g.get('output_directory', '')

effective_use_lora = bool(use_lora and float(lora_multiplier) != 0.0)
final_lora_path = ""
if effective_use_lora:
    if custom_lora_path.strip():
        final_lora_path = custom_lora_path.strip()
    elif output_directory and os.path.exists(output_directory):
        lora_files = glob.glob(os.path.join(output_directory, "*.safetensors"))
        if lora_files:
            final_lora_path = max(lora_files, key=os.path.getmtime)
            print(f"🔍 Auto-detected latest LoRA: {final_lora_path}")
    if not final_lora_path or not os.path.exists(final_lora_path):
        print(f"⚠ LoRA not found ({final_lora_path or 'none'}) — generating with base model only.")
        effective_use_lora = False

if not DIT_MODEL or not os.path.exists(DIT_MODEL):
    print(f"❌ DiT Model not found at: {DIT_MODEL}. 请先运行 Training Settings。")
else:
    out_dir = "/content/base_model_samples"
    os.makedirs(out_dir, exist_ok=True)

    n = max(1, int(num_images))
    seeds = [seed + i for i in range(n)] if seed != -1 else [random.randint(1, 999999999) for _ in range(n)]
    prompt_file = "/content/infer_prompts.txt"
    with open(prompt_file, "w", encoding="utf-8") as f:
        for s in seeds:
            f.write(f"{prompt} --n {negative_prompt} --w {width} --h {height} --s {infer_steps} --l {cfg_scale} --d {s}\n")

    cmd = [
        "python", "/content/sd-scripts/anima_minimal_inference.py",
        "--dit", DIT_MODEL, "--text_encoder", QWEN3_MODEL, "--vae", VAE_MODEL,
        "--from_file", prompt_file,
        "--image_size", str(height), str(width),
        "--save_path", out_dir,
        "--attn_mode", "torch",
    ]
    cmd += ["--qwen_image_vae_2d"] if USE_2D_VAE else ["--vae_chunk_size", "64", "--vae_disable_cache"]
    if use_fp8:
        cmd.append("--fp8")
    if effective_use_lora:
        cmd += ["--lora_weight", final_lora_path, "--lora_multiplier", str(lora_multiplier)]

    print(f"DiT:   {DIT_MODEL}")
    print(f"LoRA:  {final_lora_path + f' (×{lora_multiplier})' if effective_use_lora else 'No'}")
    print(f"Size:  {width}x{height}, steps {infer_steps}, cfg {cfg_scale}, {n} images, seeds {seeds}")
    print("-" * 40)

    before = set(glob.glob(os.path.join(out_dir, "*.png")))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            cwd="/content/sd-scripts")
    try:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        proc.kill(); proc.wait(); raise
    proc.wait()

    new_imgs = sorted(set(glob.glob(os.path.join(out_dir, "*.png"))) - before, key=os.path.getmtime)
    if new_imgs:
        for p in new_imgs:
            print(f"✓ {p}")
            display(Image(filename=p, width=384))
    if proc.returncode != 0:
        print(f"❌ Generation failed (exit code {proc.returncode}).")

### 📦 批量生成正则图像并打包导出 Zip
用底模（不加 LoRA）按提示词列表生成图片，每张配一个同名 `.txt`，最后打包下载。v6 起模型只加载一次。

In [ ]:
#@title 🚀 批量生成与 Zip 打包导出（随机尺寸）
import os, glob, random, shutil, subprocess
from IPython.display import Image, display
from google.colab import files

PROMPT_ARRAY = [
    "masterpiece, best quality, score_9, score_8, 1boy, handsome male, traditional Chinese hanfu, playing ancient guqin under a blooming cherry blossom tree, mountain peak at sunrise, soft wind, flowing white robes, serene expression",
    "masterpiece, best quality, score_9, score_8, 1girl, solo, wearing futuristic neon-lined tactical jacket, glowing short hair, standing in a rainy cyberpunk alleyway, reflection of neon billboards in puddles, cinematic night lighting",
    "masterpiece, best quality, score_9, score_8, 1boy, cool male hacker, wearing techwear, multiple holographic screens floating around him, dark high-tech room, cyan and purple accent lighting, intense focused expression",
    "masterpiece, best quality, score_9, score_8, cute fluffy little fox with nine tails, glowing fireflies in a mystical deep forest at night, magical glowing mushrooms on the ground, soft whimsical watercolor style",
    "masterpiece, best quality, score_9, score_8, scenic, majestic floating islands in a sea of clouds, giant waterfall cascading into the abyss, ancient fantasy temple, bright sunny day, cinematic scale, epic landscape",
    "masterpiece, best quality, score_9, score_8, 1boy, wearing ornate silver armor, standing on a windy cliff, dramatic dark clouds, holding a glowing magical sword pointing to the sky, epic pose",
    "masterpiece, best quality, score_9, score_8, chibi girl and chibi boy sitting together on a grassy hill, eating delicious strawberry cake, soft warm lighting, cute and heartwarming, highly detailed pastel drawing",
    "masterpiece, best quality, score_9, score_8, futuristic cyberpunk city street, flying cars passing between towering skyscrapers, neon signs in various languages, bustling night life, rainy weather, ultra detailed architecture",
    "masterpiece, best quality, score_9, score_8, 1girl, flowing translucent blue dress, walking on a crystalline path in space, galaxy and colorful nebula in the background, star dust floating around, ethereal lighting, surreal fantasy art",
    "masterpiece, best quality, score_9, score_8, mystical Chinese qilin, divine beast, body covered in glowing blue scales, stepping on golden clouds, mountain temple in the background, mythological art style, divine atmosphere",
]

SIZE_CANDIDATES = [(1024, 1024), (896, 1152), (1152, 896), (768, 1024), (1024, 768)]

#@markdown ### ⚙️ 基础生成参数
infer_steps = 28 #@param {type:"integer"}
cfg_scale = 5.0 #@param {type:"number"}
negative_prompt = "worst quality, low quality, score_1, score_2, score_3, artist name, blurry, jpeg artifacts, lowres, censor, cluttered background, photo, photorealistic, 3d render, digital art, flat shading" #@param {type:"string"}
export_zip_name = "regular_images_export" #@param {type:"string"}

g = globals()
DIT_MODEL   = g.get('DIT_MODEL', '/content/drive/MyDrive/ComfyUI/models/diffusion_models/waiANIMA_v10Base10.safetensors')
QWEN3_MODEL = g.get('QWEN3_MODEL', '/content/drive/MyDrive/ComfyUI/models/text_encoders/waiANIMA_v10Base10_txt.safetensors')
VAE_MODEL   = g.get('VAE_MODEL', '/content/drive/MyDrive/ComfyUI/models/vae/qwenImage_qwenImageVAE.safetensors')
USE_2D_VAE  = g.get('use_2d_vae', True)

prompts = [p.strip() for p in PROMPT_ARRAY if p.strip()]
export_dir = "/content/batch_regular_images"
gen_dir = "/content/batch_regular_raw"
for d in (export_dir, gen_dir):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

if not os.path.exists(DIT_MODEL):
    print(f"❌ 基础模型文件未找到: {DIT_MODEL}。请确保模型路径正确并已挂载云盘。")
else:
    # 每行一个任务：内联尺寸和唯一 seed，输出文件名里带 seed，据此对应回提示词
    seeds = random.sample(range(1, 999999999), len(prompts))
    jobs, lines = [], []
    for idx, (p, s) in enumerate(zip(prompts, seeds)):
        w, h = random.choice(SIZE_CANDIDATES)
        jobs.append((f"regular_img_{idx+1:03d}_{w}x{h}_{s}", p, s))
        lines.append(f"{p} --n {negative_prompt} --w {w} --h {h} --s {infer_steps} --l {cfg_scale} --d {s}\n")
    prompt_file = "/content/batch_prompts.txt"
    with open(prompt_file, "w", encoding="utf-8") as f:
        f.writelines(lines)
    print(f"📂 共 {len(prompts)} 个任务，模型只加载一次...")

    cmd = [
        "python", "/content/sd-scripts/anima_minimal_inference.py",
        "--dit", DIT_MODEL, "--text_encoder", QWEN3_MODEL, "--vae", VAE_MODEL,
        "--from_file", prompt_file,
        "--image_size", "1024", "1024",
        "--save_path", gen_dir,
        "--attn_mode", "torch",
    ]
    cmd += ["--qwen_image_vae_2d"] if USE_2D_VAE else ["--vae_chunk_size", "64", "--vae_disable_cache"]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            cwd="/content/sd-scripts")
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()

    ok = 0
    for name, p, s in jobs:
        hits = glob.glob(os.path.join(gen_dir, f"*_{s}.png"))
        if hits:
            final_img = os.path.join(export_dir, f"{name}.png")
            shutil.move(hits[0], final_img)
            with open(os.path.join(export_dir, f"{name}.txt"), "w", encoding="utf-8") as f:
                f.write(p)
            ok += 1
            print(f"✓ {final_img}")
            display(Image(filename=final_img, width=300))
        else:
            print(f"❌ 未生成: {name}")

    if ok:
        zip_base = f"/content/{export_zip_name}"
        shutil.make_archive(zip_base, 'zip', export_dir)
        print(f"\n📦 已打包 {ok} 张图及提示词 → {zip_base}.zip")
        files.download(f"{zip_base}.zip")
    else:
        print("\n⚠ 没有成功生成的图像，跳过打包导出。")